In [7]:
import os
import polars as pl
from pyproj import Transformer
from collections import defaultdict

BASE_DIR   = "../../data"
OUTPUT_DIR = os.path.join(BASE_DIR, "BASt", "AggregatedDataForKiel")
YEARS    = [2021, 2022, 2023, 2024, 2025]
MONTHS   = [f"{m:02d}" for m in range(1, 13)]
VARIANTS = ["A", "B"]

def load_bounding_box():
    df = pl.read_csv(os.path.join(BASE_DIR, "BoundingBox", "kiel_bb.csv"))
    return df.row(0, named=True)

def get_kiel_stations(meta_file, bb, transformer):
    stations = []
    for row in pl.read_csv(meta_file, infer_schema_length=0).iter_rows(named=True):
        try:
            e = float(str(row["Koordinaten_UTM32_E"]).replace(".", "").replace(",", "."))
            n = float(str(row["Koordinaten_UTM32_N"]).replace(".", "").replace(",", "."))
            lat, lon = transformer.transform(e, n)
            if bb["lat_min"] <= lat <= bb["lat_max"] and bb["lon_min"] <= lon <= bb["lon_max"]:
                stations.append(int(row["Dauerzaehlstellennummer"]))
        except Exception:
            pass
    return stations

def main():
    bb = load_bounding_box()
    transformer = Transformer.from_crs("EPSG:25832", "EPSG:4326", always_xy=False)
    station_months = defaultdict(set)
    filtered_data  = []

    for year in YEARS:
        for variant in VARIANTS:
            folder = os.path.join(BASE_DIR, "BASt", "AggregatedDataForSH", f"Data {year}", f"SH_{year}_{variant}_S")
            for month in MONTHS:
                meta_file = os.path.join(folder, f"Meta_{year}_{month}_{variant}_S.csv")
                roh_file  = os.path.join(folder, f"Roh_{year}_{month}_{variant}_S.csv")

                if not os.path.exists(meta_file) or not os.path.exists(roh_file):
                    continue

                stations = get_kiel_stations(meta_file, bb, transformer)
                for nr in stations:
                    station_months[nr].add(f"{year}-{month}")

                if stations:
                    filtered_data.append(
                        pl.read_csv(roh_file, infer_schema_length=0)
                        .with_columns(pl.col("Zst").str.strip_chars().cast(pl.Int64, strict=False).alias("Zst_int"))
                        .filter(pl.col("Zst_int").is_in(stations))
                        .drop("Zst_int")
                        .with_columns(pl.lit(year).alias("year"), pl.lit(month).alias("month"), pl.lit(variant).alias("variant"))
                    )

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    pl.concat(filtered_data, how="diagonal").write_csv(os.path.join(OUTPUT_DIR, "kiel_stations_raw.csv"))
    pl.DataFrame([
        {"station": nr, "months_present": len(m), "months": ", ".join(sorted(m))}
        for nr, m in sorted(station_months.items())
    ]).write_csv(os.path.join(OUTPUT_DIR, "kiel_station_count.csv"))

    return station_months

if __name__ == "__main__":
    main()